# 01 · Data Exploration — CERTIFY-BTC

Phase 1 sanity check for the **Nickparvar** brain-tumor MRI dataset. This notebook:
1. counts images per class in `Training/` and `Testing/` (the *full* dataset, ignoring the local sample cap),
2. shows raw sample images per class,
3. visualizes what preprocessing (CLAHE + resize + normalize) actually does,
4. pulls one real batch from the DataLoader to confirm shapes.

> Runs on the RTX 4050 in `MACHINE="local"` mode. Nothing here trains — it's verification only.


In [ ]:
# --- Setup: find the project root so imports + config work no matter where the notebook opens ---
import os, sys

def find_root(start):
    p = os.path.abspath(start)
    while p != os.path.dirname(p):
        if os.path.exists(os.path.join(p, "config.py")):
            return p
        p = os.path.dirname(p)
    raise RuntimeError("Could not find project root (config.py). Open this from inside the repo.")

ROOT = find_root(os.getcwd())
os.chdir(ROOT)
sys.path.insert(0, ROOT)

import glob
import numpy as np
import matplotlib.pyplot as plt

import config
from modules.preprocessing import preprocess_image, apply_clahe, denormalize
from modules.datasets import scan_split, build_dataloaders

config.summary()


## 1 · Class counts on disk (full dataset)

In [ ]:
# Count files directly on disk so we see the TRUE distribution, independent of
# config.LIMIT_SAMPLES (which only shrinks the DataLoaders, not the folders).
ds = "nickparvar"
root = config.DATASET_PATHS[ds]

counts = {"Training": {}, "Testing": {}}
for split in ("Training", "Testing"):
    for cls in config.CLASS_NAMES:
        n = len(glob.glob(os.path.join(root, split, cls, "*.jpg")))
        counts[split][cls] = n

print(f"{'class':<12}{'Training':>10}{'Testing':>10}")
for cls in config.CLASS_NAMES:
    print(f"{cls:<12}{counts['Training'][cls]:>10}{counts['Testing'][cls]:>10}")
print(f"{'TOTAL':<12}{sum(counts['Training'].values()):>10}{sum(counts['Testing'].values()):>10}")


In [ ]:
# Bar chart of the class distribution.
x = np.arange(len(config.CLASS_NAMES))
w = 0.38
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - w/2, [counts['Training'][c] for c in config.CLASS_NAMES], w, label='Training')
ax.bar(x + w/2, [counts['Testing'][c] for c in config.CLASS_NAMES], w, label='Testing')
ax.set_xticks(x); ax.set_xticklabels(config.CLASS_NAMES)
ax.set_ylabel('images'); ax.set_title('Nickparvar class distribution')
ax.legend(); plt.tight_layout(); plt.show()


## 2 · Raw sample images

Four random samples per class, as stored on disk.

In [ ]:
import cv2
rng = np.random.default_rng(config.SEED)

fig, axes = plt.subplots(len(config.CLASS_NAMES), 4, figsize=(10, 10))
for r, cls in enumerate(config.CLASS_NAMES):
    files = glob.glob(os.path.join(root, "Training", cls, "*.jpg"))
    picks = rng.choice(files, size=4, replace=False)
    for c, f in enumerate(picks):
        img = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
        axes[r, c].imshow(img, cmap="gray")
        axes[r, c].axis("off")
        if c == 0:
            axes[r, c].set_ylabel(cls, rotation=0, ha="right", va="center", fontsize=11)
            axes[r, c].axis("on"); axes[r, c].set_xticks([]); axes[r, c].set_yticks([])
plt.suptitle("Raw MRI samples per class", y=0.92)
plt.tight_layout(); plt.show()


## 3 · What preprocessing does

Left→right: raw grayscale → after CLAHE (local contrast) → final tensor the model sees
(resized to 380×380 and ImageNet-normalized, shown de-normalized for display).

In [ ]:
sample = glob.glob(os.path.join(root, "Training", "glioma", "*.jpg"))[0]
raw = cv2.imread(sample, cv2.IMREAD_GRAYSCALE)
clahe_img = apply_clahe(raw)
final = denormalize(preprocess_image(sample, use_clahe=True))  # HxWx3 in [0,1]

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(raw, cmap="gray");       ax[0].set_title(f"raw  {raw.shape}");            ax[0].axis("off")
ax[1].imshow(clahe_img, cmap="gray"); ax[1].set_title("+ CLAHE");                       ax[1].axis("off")
ax[2].imshow(final);                  ax[2].set_title(f"model input  {final.shape[:2]}"); ax[2].axis("off")
plt.tight_layout(); plt.show()


## 4 · One real batch from the DataLoader

Confirms shapes and that augmentation/normalization run end-to-end.

In [ ]:
import torch
train_loader, val_loader, test_loader = build_dataloaders()

xb, yb = next(iter(train_loader))
print("batch images:", tuple(xb.shape), xb.dtype)
print("batch labels:", yb.tolist(), "->", [config.CLASS_NAMES[i] for i in yb.tolist()])

# Show the batch (de-normalized so it's viewable).
n = xb.shape[0]
fig, ax = plt.subplots(1, n, figsize=(3*n, 3))
ax = np.atleast_1d(ax)
for i in range(n):
    ax[i].imshow(denormalize(xb[i]))
    ax[i].set_title(config.CLASS_NAMES[yb[i].item()]); ax[i].axis("off")
plt.suptitle("One augmented training batch (what the model sees)")
plt.tight_layout(); plt.show()


---
✅ **Phase 1 verified** if: counts look right, images render, CLAHE visibly boosts contrast,
and the batch is `(B, 3, 380, 380)`. Next up — **Phase 2**: `cbam.py` + `model.py`.